# Airplane Distribution Problem with AMPL in Python
## AMPL Modeling for Book Problem 3.4

This notebook models the airplane-loading problem from book section `3.4` with AMPL from Python using `amplpy`.


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [1]:
from __future__ import annotations

from functools import wraps
from math import isclose
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


def create_ampl_instance(solver: str = "highs"):
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl


def round_if_close(value: float, digits: int = 4):
    rounded = round(value, digits)
    return int(round(rounded)) if isclose(rounded, round(rounded), abs_tol=1e-9) else rounded


def extract_2d_solution(variable, row_keys, col_keys):
    raw = variable.get_values().to_dict()
    values = {}
    for row in row_keys:
        for col in col_keys:
            values[(row, col)] = round_if_close(float(raw[(row, col)]))
    return values


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Problem: Airplane Loading with Equal Weight Ratios

The aircraft has three compartments with separate weight and volume capacities. The model maximizes total utility from transported cargo while forcing every compartment to use the same fraction of its weight capacity.


In [3]:
CARGO_TYPES = ["tools", "books", "flowers", "sculptures"]
COMPARTMENTS = ["front", "center", "rear"]

WEIGHT_CAPACITY = {"front": 16, "center": 20, "rear": 14}
VOLUME_CAPACITY = {"front": 200, "center": 250, "rear": 150}
AVAILABLE = {"tools": 20, "books": 15, "flowers": 8, "sculptures": 10}
VOLUME_PER_TON = {"tools": 1, "books": 2, "flowers": 10, "sculptures": 6}
UTILITY_PER_TON = {"tools": 250, "books": 280, "flowers": 500, "sculptures": 360}

EXPECTED_TOTALS = {"tools": 17, "books": 15, "flowers": 8, "sculptures": 10}
EXPECTED_UTILITY = 16_050
EXPECTED_G = 1.0


@timed
def solve_airplane_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set CARGO ordered;
        set COMPARTMENTS ordered;

        param weight_capacity {COMPARTMENTS} >= 0;
        param volume_capacity {COMPARTMENTS} >= 0;
        param available {CARGO} >= 0;
        param volume_per_ton {CARGO} >= 0;
        param utility_per_ton {CARGO} >= 0;

        var Load {c in CARGO, k in COMPARTMENTS} >= 0;
        var G >= 0;

        maximize TotalUtility:
            sum {c in CARGO, k in COMPARTMENTS} utility_per_ton[c] * Load[c, k];

        subject to WeightCapacity {k in COMPARTMENTS}:
            sum {c in CARGO} Load[c, k] <= weight_capacity[k];

        subject to VolumeCapacity {k in COMPARTMENTS}:
            sum {c in CARGO} volume_per_ton[c] * Load[c, k] <= volume_capacity[k];

        subject to Availability {c in CARGO}:
            sum {k in COMPARTMENTS} Load[c, k] <= available[c];

        subject to EqualWeightRatio {k in COMPARTMENTS}:
            sum {c in CARGO} Load[c, k] / weight_capacity[k] = G;
        '''
    )
    ampl.set["CARGO"] = CARGO_TYPES
    ampl.set["COMPARTMENTS"] = COMPARTMENTS
    ampl.param["weight_capacity"] = WEIGHT_CAPACITY
    ampl.param["volume_capacity"] = VOLUME_CAPACITY
    ampl.param["available"] = AVAILABLE
    ampl.param["volume_per_ton"] = VOLUME_PER_TON
    ampl.param["utility_per_ton"] = UTILITY_PER_TON
    ampl.solve()

    allocation = extract_2d_solution(ampl.var["Load"], CARGO_TYPES, COMPARTMENTS)
    transported_by_cargo = {
        cargo: round_if_close(sum(allocation[(cargo, compartment)] for compartment in COMPARTMENTS))
        for cargo in CARGO_TYPES
    }
    compartment_loads = {
        compartment: round_if_close(sum(allocation[(cargo, compartment)] for cargo in CARGO_TYPES))
        for compartment in COMPARTMENTS
    }
    return {
        "allocation": allocation,
        "transported_by_cargo": transported_by_cargo,
        "compartment_loads": compartment_loads,
        "G": round_if_close(ampl.var["G"].value()),
        "utility": round_if_close(ampl.obj["TotalUtility"].value()),
    }


ampl_result, ampl_time = solve_airplane_with_ampl()
print("=== AIRPLANE DISTRIBUTION RESULTS WITH AMPL ===")
print(f"Solution -> {ampl_result}")
print(f"Time     -> {ampl_time:.8f} seconds")

assert ampl_result["utility"] == EXPECTED_UTILITY
assert ampl_result["transported_by_cargo"] == EXPECTED_TOTALS
assert abs(ampl_result["G"] - EXPECTED_G) <= 1e-9
print("AMPL matches the published utility and the unique optimal cargo totals.")


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === AIRPLANE DISTRIBUTION RESULTS WITH AMPL ===
Solution -> {'allocation': {('tools', 'front'): 0, ('tools', 'center'): 11, ('tools', 'rear'): 6, ('books', 'front'): 6, ('books', 'center'): 9, ('books', 'rear'): 0, ('flowers', 'front'): 0, ('flowers', 'center'): 0, ('flowers', 'rear'): 8, ('sculptures', 'front'): 10, ('sculptures', 'center'): 0, ('sculptures', 'rear'): 0}, 'transported_by_cargo': {'tools': 17, 'books': 15, 'flowers': 8, 'sculptures': 10}, 'compartment_loads': {'front': 16, 'center': 20, 'rear': 14}, 'G': 1, 'utility': 16050}
Time     -> 1.74330525 seconds
AMPL matches the published utility and the unique optimal cargo totals.


## Note on Multiple Optimal Layouts

The total transported tonnage by cargo type is unique, but the exact compartment assignment is not. Different allocations may produce the same utility as long as all compartments remain fully loaded with the same weight ratio.
